## <center> Prelims 2024
### <center> Mani Tyagi
#### <center> 200973108

In [61]:
# importing libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import gmres

### Part (b)

In [62]:
# function to define basis function phi
def basis_functions(x, c):
    """Define basis functions."""
    return np.exp(-1000 * (x - c)**2)

In [63]:
# function to create X matrix
def create_X(x, c_values):
    """Create the matrix X."""
    N = len(x)
    L = len(c_values)
    X = np.zeros((N, L))
    for i in range(L):
        X[:, i] = basis_functions(x, c_values[i])
    return X

# testing the function
x = np.linspace(0, 1, 4)
c = np.linspace(0, 1, 4)
X = create_X(x, c)
print("Matrix A:\n", X)

Matrix A:
 [[1.00000000e+000 5.55977948e-049 9.55499062e-194 0.00000000e+000]
 [5.55977948e-049 1.00000000e+000 5.55977948e-049 9.55499062e-194]
 [9.55499062e-194 5.55977948e-049 1.00000000e+000 5.55977948e-049]
 [0.00000000e+000 9.55499062e-194 5.55977948e-049 1.00000000e+000]]


In [64]:
# solving the least squares problem
def solve_least_squares(X, y, lambda_val):
    """Solve the regularized least squares problem."""
    L = X.shape[1]  # Number of basis functions
    I = np.eye(L)
    w = np.linalg.solve(X.T @ X + lambda_val * I, X.T @ y)
    return w

In [65]:
def evaluate_model(X, y, w):
    """Evaluate the model."""
    error = np.linalg.norm(y - X @ w)**2
    weight_norm = np.linalg.norm(w)**2
    return error, weight_norm

In [66]:
# Generate data points
N = 100
x = np.linspace(0, 1, N)
y = np.sin(2 * np.pi * x) + 0.1*np.sin(20*np.pi*x)

# Define basis functions
c_values = np.linspace(0, 1, 100)
X = create_X(x, c_values)

# Set regularization parameter
lambda_val = 0.01

# Solve the regularized least squares problem
w = solve_least_squares(X, y, lambda_val)

# Evaluate the model
error, weight_norm = evaluate_model(X, y, w)

print("Error of the model:", error)
print("Norm of the coefficients w:", weight_norm)


Error of the model: 0.0007026968626756603
Norm of the coefficients w: 1.844352235692822


### Part (c)

In [67]:
# Generate data points
N = 100
x = np.linspace(0, 1, N)
y = np.sin(2 * np.pi * x) + 0.1*np.sin(20*np.pi*x)

# Define basis functions
c_values = np.linspace(0, 1, 100)
X = create_X(x, c_values)

# Set lambda values to test
lambda_values = [0.0, 1e-8, 1e-6, 1e-4, 1e-2, 1, 100]

# Initialize lists to store results
results = []

# Loop over lambda values
for lambda_val in lambda_values:
    # Solve the regularized least squares problem
    w = solve_least_squares(X, y, lambda_val)
    
    # Evaluate the model
    error, weight_norm = evaluate_model(X, y, w)
    
    # Store results
    results.append([lambda_val, error, weight_norm])

# Printing results
print("-" * 55)
print("{:<15} {:<15} {:<25}".format("Lambda", "Error", "Norm of Coefficients"))
print("-" * 55)
for result in results:
    print("{:<15g} {:<15.6f} {:<25.6f}".format(result[0], result[1], result[2]))
print("-" * 55)


-------------------------------------------------------
Lambda          Error           Norm of Coefficients     
-------------------------------------------------------
0               0.000000        29106.743685             
1e-08           0.000000        40.597778                
1e-06           0.000004        6.777142                 
0.0001          0.000052        2.417206                 
0.01            0.000703        1.844352                 
1               0.080763        1.611000                 
100             29.642011       0.088426                 
-------------------------------------------------------


### Part (d)

In [72]:
def solve_gmres(X, y, lambda_val, tol=1e-12, max_iter=10000):
    """Solve the regularized least squares problem using GMRES."""
    L = X.shape[1]  # Number of basis functions
    I = np.eye(L)
    A = X.T @ X + lambda_val * I
    b = X.T @ y
    w, info = gmres(A, b, tol=tol, maxiter=max_iter)
    
    if isinstance(info, int):
        # If GMRES did not return additional information, set iterations to info
        iterations = info
        exit_code = "Convergence"
    else:
        # If GMRES returned a tuple with additional information, extract the iteration count and exit code
        iterations = info[1]
        if info[1] == 0:
            exit_code = "Did not converge"
        else:
            exit_code = "Convergence"
    
    return w, iterations, exit_code


In [73]:
# Generate data points
N = 100
x = np.linspace(0, 1, N)
y = np.sin(2 * np.pi * x) + 0.1*np.sin(20*np.pi*x)

# Define basis functions
c_values = np.linspace(0, 1, 100)
X = create_X(x, c_values)

# Set lambda values to test
lambda_values = [0.0, 1e-8, 1e-6, 1e-4, 1e-2, 1, 100]

# Initialize list to store results
results = []

# Loop over lambda values
for lambda_val in lambda_values:
    # Solve the regularized least squares problem
    w = solve_least_squares(X, y, lambda_val)
    
    # Evaluate the model
    error, weight_norm = evaluate_model(X, y, w)
    
    # Solve the regularized least squares problem using GMRES
    w_gmres, iterations, exit_code = solve_gmres(X, y, lambda_val)
    
    # Store results
    results.append([lambda_val, error, weight_norm, iterations, exit_code])

print("-" * 90)
print("{:<15} {:<15} {:<25} {:<15} {:<25}".format("Lambda", "Error", "Norm of Coefficients", "Iterations", "Convergence"))
print("-" * 90)
for result in results:
    print("{:<15g} {:<15.6f} {:<25.6f} {:<15} {:<25}".format(result[0], result[1], result[2], result[3], result[4]))
print("-" * 90)


------------------------------------------------------------------------------------------
Lambda          Error           Norm of Coefficients      Iterations      Convergence              
------------------------------------------------------------------------------------------
0               0.000000        29106.743685              10000           Convergence              
1e-08           0.000000        40.597778                 10000           Convergence              
1e-06           0.000004        6.777142                  10000           Convergence              
0.0001          0.000052        2.417206                  0               Convergence              
0.01            0.000703        1.844352                  0               Convergence              
1               0.080763        1.611000                  0               Convergence              
100             29.642011       0.088426                  0               Convergence              
------------------